# Stage 1: Load the dataset and prepare the data

### Steps to prepare the data:

1. Raw text

2. Token IDs

3. Input/target windows

4. Dataset and DataLoader

5. Token embeddings

6. Positional embeddings

7. Final input embeddings

In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.11.0


## 1. Load the raw text

In [13]:
from pathlib import Path

data_path = Path("../../../data/the-verdict.txt")
text = data_path.read_text(encoding="utf-8")

print(text[:100])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


## 2. Generate token ids

Future work: implement a BPE tokenizer from scratch

By now, it's enough by using tokenizer library

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [30]:
text_spanish = "Hola, hoy voy a visitar a mi familia en la ciudad."
text_english = "Hello, today I am going to visit my family at the city."

token_ids_sp = tokenizer.encode(text_spanish)
token_ids_en = tokenizer.encode(text_english)

print(len(token_ids_sp))
print(len(token_ids_en))

20
14


*Language Tax:* Just discovered why using LLMs in English reduces costs generally.

In [36]:
token_ids = tokenizer.encode(text)

reconstructed = tokenizer.decode(token_ids)

assert reconstructed == text

## 3. Dataset & Dataloader

First, we need to create the window for create the pairs input -- target

In [39]:
max_length = 4

input_chunk = token_ids[:max_length]
target_chunk = token_ids[1:max_length+1]

print(input_chunk)
print(target_chunk)

[40, 367, 2885, 1464]
[367, 2885, 1464, 1807]


In [40]:
print(tokenizer.decode(input_chunk))
print(tokenizer.decode(target_chunk))

I HAD always
 HAD always thought


Next, create the batch of windows

In [41]:
max_length = 4
stride = 4

for i in range(0, len(token_ids) - max_length, stride):
    input_chunk = token_ids[i : i + max_length]
    target_chunk = token_ids[i + 1 : i + max_length + 1]

    print("Input: ", tokenizer.decode(input_chunk))
    print("Target:", tokenizer.decode(target_chunk))
    print()

Input:  I HAD always
Target:  HAD always thought

Input:   thought Jack Gis
Target:  Jack Gisburn

Input:  burn rather a cheap
Target:  rather a cheap genius

Input:   genius--though a
Target: --though a good

Input:   good fellow enough--
Target:  fellow enough--so

Input:  so it was no
Target:  it was no great

Input:   great surprise to me
Target:  surprise to me to

Input:   to hear that,
Target:  hear that, in

Input:   in the height of
Target:  the height of his

Input:   his glory, he
Target:  glory, he had

Input:   had dropped his painting
Target:  dropped his painting,

Input:  , married a rich
Target:  married a rich widow

Input:   widow, and established
Target: , and established himself

Input:   himself in a vill
Target:  in a villa

Input:  a on the Riv
Target:  on the Riviera

Input:  iera. (Though
Target: . (Though I

Input:   I rather thought it
Target:  rather thought it would

Input:   would have been Rome
Target:  have been Rome or

Input:   or Florence.)

Target: 

Now, implement the Dataset for having the data to train.

This will implement all the stuff from above lines

In [42]:
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        token_ids = tokenizer.encode(text)

        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))


    def __len__(self):
        return(len(self.input_ids))

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [43]:
dataset = GPTDatasetV1(
    text=text,
    tokenizer=tokenizer,
    max_length=4,
    stride=4,
)

x, y = dataset[0]

print(x)
print(y)
print(x.shape, y.shape)
print(x.dtype, y.dtype)

tensor([  40,  367, 2885, 1464])
tensor([ 367, 2885, 1464, 1807])
torch.Size([4]) torch.Size([4])
torch.int64 torch.int64


Now, we implement the DataLoader to load the data

In [45]:
from torch.utils.data import DataLoader

def create_data_loader_v1(txt, batch_size=4, max_length=256, 
                          stride=128, shuffle=True, drop_last=True, 
                          num_workers=0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [66]:
dataloader = create_data_loader_v1(
    text,
    batch_size=2,
    max_length=4,
    stride=3,
    shuffle=False,
    drop_last=False,
)

inputs, targets = next(iter(dataloader))

print("Inputs:")
print(inputs)

print("\nTargets:")
print(targets)

print("\nShapes:")
print(inputs.shape)
print(targets.shape)

print("\nDtypes:")
print(inputs.dtype)
print(targets.dtype)

Inputs:
tensor([[  40,  367, 2885, 1464],
        [1464, 1807, 3619,  402]])

Targets:
tensor([[ 367, 2885, 1464, 1807],
        [1807, 3619,  402,  271]])

Shapes:
torch.Size([2, 4])
torch.Size([2, 4])

Dtypes:
torch.int64
torch.int64


## 4. Generate token embeddings

In [62]:
import torch.nn as nn

vocab_size = 6
embedding_dim = 3

torch.manual_seed(123)

token_embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

token_ids = torch.tensor([2, 3, 1])

embeddings = token_embedding(token_ids)

print(token_ids.shape)
print(embeddings.shape)
print(embeddings)

torch.Size([3])
torch.Size([3, 3])
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [59]:
print(token_embedding.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [71]:
vocab_size = 50257
embedding_dim = 256

token_embedding_layer = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
)

input_embeddings = token_embedding_layer(inputs)

print("Token IDs:", inputs.shape)
print("Token embeddings:", input_embeddings.shape)

Token IDs: torch.Size([2, 4])
Token embeddings: torch.Size([2, 4, 256])


## 5. Positional embeddings

For adding positional information, just adding to each input token its position within the batch example

In [68]:
context_length = inputs.shape[1]

pos_embedding_layer = nn.Embedding(
    num_embeddings=context_length,
    embedding_dim=embedding_dim,
)

positions = torch.arange(context_length)

print(positions)

pos_embeddings = pos_embedding_layer(positions)

print(pos_embeddings.shape)

tensor([0, 1, 2, 3])
torch.Size([4, 256])


### Input Embeddings:

So, the Final Input Embeddings are as following

In [69]:
token_embeddings = token_embedding_layer(inputs)
pos_embeddings = pos_embedding_layer(positions)

input_embeddings = input_embeddings + pos_embeddings

print(token_embeddings.shape)
print(pos_embeddings.shape)
print(input_embeddings.shape)

torch.Size([2, 4, 256])
torch.Size([4, 256])
torch.Size([2, 4, 256])


## Summary

The data preparation pipeline transforms raw text into numerical
representations that can be processed by the model:

1. The tokenizer converts raw text into token IDs.
2. Input-target pairs are created using sliding windows.
3. Targets contain the same tokens as the inputs shifted one position forward.
4. A `Dataset` stores individual input-target pairs with shape `(T,)`.
5. A `DataLoader` groups multiple examples into batches with shape `(B, T)`.
6. Token embeddings replace every token ID with a trainable vector of size `C`,
   transforming `(B, T)` into `(B, T, C)`.
7. Positional embeddings represent the positions within the context.
8. Token and positional embeddings are added to produce the model input.

### Main dimensions

- `V` — vocabulary size: number of possible tokens.
- `B` — batch size: number of sequences processed together.
- `T` — sequence length: number of tokens in each example.
- `C` — embedding dimension: number of features representing each token.

### Main tensor shapes

- Token IDs: `(B, T)`
- Token embedding table: `(V, C)`
- Token embeddings: `(B, T, C)`
- Positional embedding table: `(context_length, C)`
- Positional embeddings used by a sequence: `(T, C)`
- Final input embeddings: `(B, T, C)`
- Targets: `(B, T)`

The model can accept sequences where `T <= context_length`. During training,
the targets remain integer token IDs because they identify the correct
next-token class.